In [5]:
## Using the LLM models we will be automating the SQL quering and NLP analysis of the data. We will be using the Gemini LLM from Google for this purpose. We will be using the Langchain library to interact with the LLM and to create a SQL agent that can query our database and perform NLP analysis on the data.

#### Instaling LLM and api

In [2]:
pip install langchain langchain-community langchain-core

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install langchain-google-genai

  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached langchain_google_genai-4.2.1-py3-none-any.whl (66 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
   ---------------------------------------- 0.0/760.6 kB ? eta -:--:--
   --------------------------- ------------ 524.3/760.6 kB 1.9 MB/s eta 0:00:01
   --------------------------- ------------ 524.3/760.6 kB 1.9 MB/s eta 0:00:01
   --------------------------- ------------ 524.3/760.6 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 760.6/760.6 kB 685.2 kB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.5 MB ? eta -

In [1]:
## Importing necessary libraries and setting up database connection

## LLM and API setup
import os
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

## Fetching credentials from env for DataBase
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

api_key = os.getenv("GOOGLE_API_KEY")

db_url = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}' 
db = SQLDatabase.from_uri(db_url,include_tables=['anchor_50', 'fact_fundamentals', 'fact_prices', 'stock_momentum'],view_support=True)
print("These are the tables available in the SQL database")
print(db.get_usable_table_names())

## Seting up the LLM with API key -- I am using Gemini LLM
os.environ["GOOGLE_API_KEY"] = api_key
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash",temperature=0)

d:\Data_Softwares\envs\AI-Driven_Analysis\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


These are the tables available in the SQL database
['anchor_50', 'fact_fundamentals', 'fact_prices', 'stock_momentum']


In [ ]:

## Adding a custom prompt for my AI agent as a defined character for it
custom_prefix = """
You are a brutally honest quantitative hedge fund analyst. 
You must answer the user's question by writing exactly ONE SQL query. Do not use 'sql_db_schema' or 'sql_db_list_tables'.

DATABASE SCHEMA:
Table 1: `anchor_50`
Columns: ticker, company_name, sector, market_cap_cat, adj_senti_score, total_sentiment_score

Table 2: `fact_fundamentals`
Columns: asset_id, pe_ratio, peg_ratio, roe_percent, debt_to_equity

Table 3: `dimens_assets_details`
Columns: asset_id, ticker

View: `stock_moentum`
Columns: asset_id, company_name, trade_date, current_price, sma_50, sma_200, momentum_signal

CRITICAL RULES:
1. To get PEG ratio, you must JOIN `anchor_50` to `dimens_assets_details` (on ticker), and then JOIN to `fact_fundamentals` (on asset_id).
2. To check momentum, JOIN `anchor_50` to `stock_momentum` (on ticker).
3. If the user asks for "Bullish" stocks, filter where `momentum_signal` LIKE '%Bullish%'.
4. Do not sugarcoat bad metrics.
5. DISCLAIMER: End EVERY single response with this exact disclaimer: "*Disclaimer: I am an AI bot created for testing, not a SEBI-registered financial advisor. This is for educational purposes only.*"

Do your job, write the correct SQL, and give the user the cold, hard truth.
"""

executing_agent = create_sql_agent(llm,db=db,agent_type="tool-calling",verbose=True,prefix=custom_prefix)

## Testing
question = "I want to invest in a stock with high sentiment but it must be trading in a Bullish trend. Give me your top 2 recommendations from the Anchor 50 and tell me their PEG ratios."
response = executing_agent.invoke({"input": question})

## Making the output clean without the signature

raw_data = response["output"]

if isinstance(raw_data,list) and len(raw_data) > 0:
    answer = raw_data[0].get("text", str(raw_data))
elif isinstance(raw_data,str):
    answer = raw_data
else:
    answer = str(raw_data)
print("Answer fetched for the query :",question)
print(answer)

These are the tables available in the SQL database
['anchor_50', 'fact_fundamentals', 'fact_prices', 'stock_momentum']


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query_checker` with `{'query': "\nSELECT\n    a50.company_name,\n    a50.adj_senti_score,\n    ff.peg_ratio\nFROM\n    anchor_50 AS a50\nJOIN\n    stock_moentum AS sm ON a50.ticker = sm.ticker\nJOIN\n    dimens_assets_details AS dad ON a50.ticker = dad.ticker\nJOIN\n    fact_fundamentals AS ff ON dad.asset_id = ff.asset_id\nWHERE\n    sm.momentum_signal LIKE '%Bullish%'\nORDER BY\n    a50.adj_senti_score DESC\nLIMIT 2;\n"}`
responded: 


```sql
SELECT
    a50.company_name,
    a50.adj_senti_score,
    ff.peg_ratio
FROM
    anchor_50 AS a50
JOIN
    stock_moentum AS sm ON a50.ticker = sm.ticker
JOIN
    dimens_assets_details AS dad ON a50.ticker = dad.ticker
JOIN
    fact_fundamentals AS ff ON dad.asset_id = ff.asset_id
WHERE
    sm.momentum_signal LIKE '%Bullish%'
ORDER BY
    a50.adj_senti_score DESC
LIMI

In [8]:
raw_data

[{'type': 'text',
  'text': 'Here are',
  'extras': {'signature': 'CnUBvj72+yRltIq9DCFdfvIdFJ6X1IY6CFp+3dhKoIPLsp1KZKQVZhGNVolj8zaAMTa9tXFqjKQQGx0SMcTuDPTpdhS5zogV8bvFNF0+HmRwSpDtM7uOC4E1+Ud301Yj9/C0d37uuZxRUlDZojJv7BRm2h1yaIEK0QEBvj72+7PNfIamGG4BsDHFwZYHFzfooLNEymJhx/zJ7XPM9A5Ubz0Vy+wd19nBtghQUjsNNp7hs/klsB545/fIWpBnQkHUGA+ybg2inQhOuXUq9IbIwPv7oeu7+Yf9GlZDVxZgyA5gaGDAlwDIt0LajNsyTKFj7TVLYy8Ci6f5+OnSLiY4+JFX74uKHgbDrAMW6zSvK63Tr4LuOUTMeNVD9JcEaN/03C7e8q8RbL5vierAnZHm3vFbVdOOHD903w6GFhM/LtVR9+foO3NYpOv01QqHAQG+Pvb7J4vCb183zRZFgSK/9iHsPtgSUswEE9Z1Fz4sO7LACCiGwZRKU+ij9LiHSuhjQO8c3nOJyCC5qq5LWKS0XdgCQbZ7VztCxCEK5HG/cLd+rrQSfQRC8jdFoW9WlqVCmeSv8dBmheiRz8INz3vs1PbU2+Nsh1dpwZHS/1lz+PYVq6hUZwrOAQG+Pvb7GqyMlxyifVL+UX2QR4trT1T3n8KCHEiYpnMsQ5Vw/Fpb/H6d4Nc02J34sueNJ5TSH7u8rHVTMUlqmtdzc7n+XDAiNdFrI0HiFFYJHHhyNRMng2CpsGoJpFRs+o3/4gF6pw/1NweY5MhqhkRzI6CarXvtcpGbmzRvYnTqZPXvvPOYViUucVUrJvmcHaqw9qa1EgPVS3f/u6BEuUXpDwGkct0YmtkThuPIvg8GysYHqO3u1XywmGvyFPo3Wav04rCGGrGGA2IvjW9e'},
  'index': 0},
 ' your to

In [7]:
raw_data[1]

' your top 2 recommendations for stocks with high sentiment and a Bullish trend:\n\n1.  **Star Health and Allied Insurance Company Ltd.**\n    *   Adjusted Sentiment Score: 0.926\n    *   PEG Ratio: -1.54 (A negative PEG ratio often indicates negative earnings growth, which is not ideal.)\n\n2.  **Bharat Heavy Electricals Ltd.**\n    *   Adjusted Sentiment Score: 0.86545\n    *   PEG Ratio: 0.61\n\n*Disclaimer: I am an AI bot created for testing, not a SEBI-registered financial advisor. This is for educational purposes only.*'